In [2]:
# This is timport datarobot as dr
import pandas as pd
import numpy as np
from datetime import datetime
from scipy.stats import mode
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import sys, os
from pathlib import Path



# Go up one directory from "notebooks" to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

from scripts.utils import (get_variable_name,
                           count_repetitive_caseids,
                            add_prefix_except_caseid, 
                            get_dummy_variables, 
                            convert_objects_to_int64_safe,
                            drop_null_and_list,
                            aggregate_sum_by_caseid,
                            analyze_matches,
                            analyze_matches_explicit_keys, 
                            aggregate_with_value_suffix, 
                            drop_high_null_columns,
                            save_file,
                            add_value_suffix,
                            save_txt
                            ) 


# Now import works
from scripts.Wrangling import (
    categorize_columns,
    fill_missing_with_mode,
    target_encode, 
    fill_missing_with_median_coding,
    #target_encode_dataframe,
    target_encode_dataframe_map,
    #sklearn_target_encode
)

In [3]:
# import sys
# print(sys.executable)

# !{sys.executable} -m pip install category_encoders

In [4]:
key_variables =['CASEID']

# 1 - Ingest Data

In [5]:
# Modulo1632_REC21_2023 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1632/968-Modulo1632/REC21_2024.csv", low_memory=False)
# Modulo1632_REC21_2023.info(verbose = True, show_counts = True)


# Build base directory (go up one level from current /02_scripts)
base_dir = os.path.dirname(os.getcwd())   # → c:\Users\linoc\OneDrive\Encoder\03_partos
base_dir_raw =  Path(base_dir)
base_dir_raw= str(base_dir_raw.parents[1])

# Build the path to the target CSV
csv_path_Modulo1633_REC41_2024_fil = os.path.join(base_dir,"data\\interim", "Modulo1633_REC41_2024_fil_v3.csv")
csv_path_Modulo1633_REC94_2024_fil = os.path.join(base_dir,"data\\interim", "Modulo1633_REC94_2024_fil_v3.csv")

# Build the path to the target CSV
csv_path_Modulo1631_REC91_2024 = os.path.join(base_dir_raw,"01_raws", "2024", "968-Modulo1631", "968-Modulo1631", "REC91_2024.csv")
csv_path_Modulo1632_RE223132_2024= os.path.join(base_dir_raw,"01_raws", "2024", "968-Modulo1632", "968-Modulo1632", "RE223132_2024.csv")
csv_path_Modulo1635_RE516171_2024 = os.path.join(base_dir_raw,"01_raws", "2024", "968-Modulo1635", "968-Modulo1635", "RE516171_2024.csv")

# Load the dataset
Modulo1633_REC41_2024_fil = pd.read_csv(csv_path_Modulo1633_REC41_2024_fil, low_memory=False)
Modulo1633_REC94_2024_fil = pd.read_csv(csv_path_Modulo1633_REC94_2024_fil, low_memory=False)
Modulo1631_REC91_2024 = pd.read_csv(csv_path_Modulo1631_REC91_2024, low_memory=False)
Modulo1632_RE223132_2024 = pd.read_csv(csv_path_Modulo1632_RE223132_2024, low_memory=False)
Modulo1635_RE516171_2024 = pd.read_csv(csv_path_Modulo1635_RE516171_2024, low_memory=False)

Modulo1633_REC41_2024_fil.shape, Modulo1633_REC94_2024_fil.shape, Modulo1631_REC91_2024.shape, Modulo1632_RE223132_2024.shape, Modulo1635_RE516171_2024.shape

((18335, 89), (18335, 54), (37117, 343), (34252, 149), (34252, 84))

In [6]:
Modulo1635_RE516171_2024.V717.value_counts(dropna=False)

V717
0     11831
3      6727
4      5026
1      3614
6      2675
8      1695
7      1303
2      1131
9       235
98       15
Name: count, dtype: int64

In [25]:
Modulo1640_CSALUD01_2024.QS25AA.value_counts(dropna=False)

QS25AA
10    23464
1      6399
       2534
2       864
4       217
3       178
5       113
9        95
6        48
11       46
7        36
8        16
12        8
Name: count, dtype: int64

In [27]:
Modulo1632_RE223132_2024.V217.value_counts(dropna=False)   

V217
3    9952
8    7339
6    6854
2    6221
1    1445
4    1327
5    1114
Name: count, dtype: int64

In [26]:
Modulo1633_REC41_2024_fil.M45.value_counts(dropna=False)

M45
1.0    14283
NaN     3353
0.0      695
8.0        4
Name: count, dtype: int64

In [7]:
Modulo1633_REC94_2024_fil.S413.value_counts(dropna=False)

S413
1.0    12035
NaN     3353
0.0     2947
Name: count, dtype: int64

In [22]:
Modulo1635_RE516171_2024.V228.value_counts(dropna=False)

AttributeError: 'DataFrame' object has no attribute 'V228'

In [5]:
csv_path_target_final = os.path.join(base_dir,"data\\interim", "target_final.csv")

# Load the dataset
target_final = pd.read_csv(csv_path_target_final,usecols = ['CASEID','premature_flag'], low_memory=False)
target_final.shape

(18335, 2)

In [6]:
Modulo1632_RE223132_2024.CASEID.nunique()
key_variables =['CASEID']

# 2 Data Wrangling

## 2.1 csv_path_Modulo1633_REC41_2024_fil

In [1]:
Modulo1633_REC41_2024_fil.M13.value_counts()

NameError: name 'Modulo1633_REC41_2024_fil' is not defined

In [7]:
Modulo1633_REC41_2024_fil.info(verbose = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 89 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   premature_flag  18335 non-null  int64  
 1   M2A             14982 non-null  float64
 2   M2B             14982 non-null  float64
 3   M2C             14982 non-null  float64
 4   M2D             14982 non-null  float64
 5   M2E             14982 non-null  float64
 6   M2G             14982 non-null  float64
 7   M2K             14982 non-null  float64
 8   M2N             14982 non-null  float64
 9   M3A             16983 non-null  float64
 10  M3B             16983 non-null  float64
 11  M3C             16983 non-null  float64
 12  M3D             16983 non-null  float64
 13  M3E             16983 non-null  float64
 14  M3G             16983 non-null  float64
 15  M3H             16983 non-null  float64
 16  M3K             16983 non-null  float64
 17  M3N             16983 non-null 

In [8]:
# categoricas, M10, M15,M42A,M42C, M42D, M42E, M43, M44,M45, M47, M48, M60, M69

categorical_cols = ['M15', 
                    'M19A',
                     'M27',
                     'M28',
                     'M29', 
                     #'M34',
                     'M10', 
                     #'M15',
                     'M42A',
                     'M42C', 
                     'M42D', 
                     'M42E', 
                     'M43', 
                     'M44',
                     'M45', 
                     'M47', 
                     'M48', 
                     'M60', 
                     'M69',
                     ]


Modulo1633_REC41_2024_fil[categorical_cols].head()

,M15,M19A,M27,M28,M29,M10,M42A,M42C,M42D,M42E,M43,M44,M45,M47,M48,M60,M69
0,21.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,21.0
1,21.0,1.0,0.0,0.0,0.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,21.0
2,21.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,21.0
3,21.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,21.0
4,21.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,21.0


In [9]:
key_variables =['CASEID']
numeric_cols, categorical_cols, dummy_cols = categorize_columns (Modulo1633_REC41_2024_fil, key_variables, categorical_cols)

🔹 Numeric columns: 37
🔹 Categorical columns: 17
🔹 Dummy columns: 34


In [10]:
categorical_cols.append('CASEID')
df_categorical = Modulo1633_REC41_2024_fil[categorical_cols].merge(target_final, how= 'left', on ='CASEID')

df_categorical_cod, encoding_map, encoder = target_encode_dataframe_map(df_categorical,
                                                               target_col='premature_flag', id_col='CASEID')

✅ Encoded 17 categorical columns.


In [11]:
df_categorical_cod.head()

,CASEID,M27,M10,M42C,M44,M45,M60,M42D,M15,M42A,M47,M29,M42E,M69,M19A,M43,M48,M28,premature_flag
0,325503101 2,0.194256,0.202711,0.193222,0.191153,0.193667,0.192802,0.192654,0.212246,0.193124,0.190103,0.194416,0.192830,0.207502,0.190584,0.191223,0.190727,0.194176,1
1,325504701 2,0.193653,0.194756,0.191335,0.189192,0.191601,0.191533,0.190574,0.214105,0.191387,0.189233,0.193798,0.190915,0.207762,0.189582,0.189348,0.189684,0.193824,0
2,325505001 1,0.194258,0.201619,0.192934,0.190476,0.193464,0.202129,0.192000,0.213112,0.192869,0.191204,0.194418,0.192459,0.208136,0.193043,0.190539,0.192188,0.194252,0
3,325508901 2,0.194256,0.202711,0.193222,0.191153,0.193667,0.192802,0.192654,0.212246,0.193124,0.190103,0.194416,0.192830,0.207502,0.190584,0.191223,0.190727,0.194176,0
4,325509701 2,0.194258,0.201619,0.192934,0.190476,0.193464,0.193568,0.192000,0.213112,0.192869,0.191204,0.194418,0.192459,0.208136,0.193043,0.190539,0.192188,0.194252,0


In [12]:
df_categorical_cod = fill_missing_with_mode(df_categorical_cod)

In [13]:
df_numeric_cols = fill_missing_with_median_coding(Modulo1633_REC41_2024_fil[numeric_cols])

Shape after transformation: (18335, 37)


In [14]:
df_load_dummy_cols =  fill_missing_with_mode(Modulo1633_REC41_2024_fil[dummy_cols])

In [15]:
df_categorical_cod.shape, df_numeric_cols.shape, df_load_dummy_cols.shape

((18335, 19), (18335, 37), (18335, 34))

In [16]:
df_categorical_cod.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CASEID          18335 non-null  object 
 1   M27             18335 non-null  float64
 2   M10             18335 non-null  float64
 3   M42C            18335 non-null  float64
 4   M44             18335 non-null  float64
 5   M45             18335 non-null  float64
 6   M60             18335 non-null  float64
 7   M42D            18335 non-null  float64
 8   M15             18335 non-null  float64
 9   M42A            18335 non-null  float64
 10  M47             18335 non-null  float64
 11  M29             18335 non-null  float64
 12  M42E            18335 non-null  float64
 13  M69             18335 non-null  float64
 14  M19A            18335 non-null  float64
 15  M43             18335 non-null  float64
 16  M48             18335 non-null  float64
 17  M28             18335 non-null 

In [17]:
Modulo1633_REC41_2024_fil_clear = pd.concat([df_numeric_cols,
                                              df_load_dummy_cols, 
                                              df_categorical_cod], axis=1)
Modulo1633_REC41_2024_fil_clear.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 90 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   BIDX_x          18335 non-null  float64
 1   ID1             18335 non-null  float64
 2   MIDX            18335 non-null  float64
 3   M1              18335 non-null  float64
 4   M1A             18335 non-null  float64
 5   M1B             18335 non-null  float64
 6   M1D             18335 non-null  float64
 7   M5              18335 non-null  float64
 8   M18             18335 non-null  float64
 9   M35             18335 non-null  float64
 10  M36             18335 non-null  float64
 11  M38             18335 non-null  float64
 12  M39             18335 non-null  float64
 13  M55A            18335 non-null  float64
 14  M55B            18335 non-null  float64
 15  M55C            18335 non-null  float64
 16  M55E            18335 non-null  float64
 17  M55F            18335 non-null 

In [18]:
save_file(Modulo1633_REC41_2024_fil_clear, output_file = "Modulo1633_REC41_2024_fil_clear_v3.csv")
save_txt(encoding_map, dir="data\\interim",output_file="encoding_map_REC41.txt")

✅ File saved successfully at: c:\Users\linoc\OneDrive\Encoder\03_partos\02_scripts\Premature_model\data\interim\encoding_map_REC41.txt


## 3.2 Modulo1633_REC94_2024_fil

In [19]:
Modulo1633_REC94_2024_fil.info(verbose = True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 54 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   premature_flag  18335 non-null  int64  
 1   S413            14982 non-null  float64
 2   S426E           6081 non-null   float64
 3   S426GA          14982 non-null  float64
 4   S426GB          14982 non-null  float64
 5   S426GC          14982 non-null  float64
 6   S426GD          14982 non-null  float64
 7   S426GE          14982 non-null  float64
 8   S430D           15095 non-null  float64
 9   S427DA          14982 non-null  float64
 10  S427DB          14982 non-null  float64
 11  S427DC          14982 non-null  float64
 12  S427DD          14982 non-null  float64
 13  S427DE          14982 non-null  float64
 14  S427DF          14982 non-null  float64
 15  S427DG          14982 non-null  float64
 16  S427F           4837 non-null   float64
 17  S436C           16835 non-null 

In [20]:
categorical_cols_REC94 = ['S411B',
                    'S411F',
                    'S411G',
                    'S411H',
                    'S411I',
                    'S411J',
                    'S411K',
                    'S411L',
                    'S426B',
                    #'S426FA',
                    'S426FB',
                    'QI411_M',
                    'QI422A_A',
                    'QI422A_B', 
                    'QI422A_C', 
                    'QI422A_D'
                    ]


In [21]:
numeric_cols_REC94, categorical_cols_REC94, dummy_cols_REC94 = categorize_columns (Modulo1633_REC94_2024_fil, key_variables,categorical_cols_REC94)
dummy_cols_REC94.remove('premature_flag')

🔹 Numeric columns: 19
🔹 Categorical columns: 15
🔹 Dummy columns: 19


In [22]:
Modulo1633_REC94_2024_fil[categorical_cols_REC94].head()

,S411L,QI422A_C,QI422A_A,S426FB,QI422A_D,S411I,S411K,S411H,S411F,QI422A_B,QI411_M,S411J,S411G,S411B,S426B
0,1.0,NaN,1.0,100.0,NaN,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,NaN
1,1.0,NaN,1.0,100.0,NaN,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,NaN
2,1.0,NaN,1.0,100.0,NaN,1.0,0.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,NaN
3,1.0,NaN,1.0,100.0,NaN,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,NaN
4,1.0,NaN,1.0,101.0,NaN,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,NaN


In [23]:
categorical_cols_REC94.append('CASEID')

df_categorical_rec94 = Modulo1633_REC94_2024_fil[categorical_cols_REC94].merge(target_final, how= 'left', on ='CASEID')

df_categorical_cod_rec94, encoding_map_rec94, encoder_rec94 = target_encode_dataframe_map(df_categorical_rec94,
                                                               target_col='premature_flag', id_col='CASEID')

df_categorical_cod_rec94.head()

✅ Encoded 15 categorical columns.


,CASEID,S411L,QI422A_C,QI422A_A,S426FB,QI422A_D,S411I,S411K,S411H,S411F,QI422A_B,QI411_M,S411J,S411G,S411B,S426B,premature_flag
0,325503101 2,0.179648,0.190885,0.192052,0.180430,0.191104,0.191480,0.180320,0.193857,0.192963,0.189430,0.190821,0.189523,0.193312,0.192650,0.182181,1
1,325504701 2,0.176358,0.189942,0.190390,0.180148,0.190191,0.189877,0.178021,0.191249,0.191272,0.185641,0.189266,0.186616,0.190622,0.191057,0.183914,0
2,325505001 1,0.180461,0.190426,0.191403,0.180828,0.190980,0.191627,0.222075,0.193476,0.193077,0.187741,0.190112,0.189115,0.191802,0.192679,0.182866,0
3,325508901 2,0.179648,0.190885,0.192052,0.180430,0.191104,0.191480,0.180320,0.193857,0.192963,0.189430,0.190821,0.189523,0.193312,0.192650,0.182181,0
4,325509701 2,0.180461,0.190426,0.191403,0.203308,0.190980,0.191627,0.182156,0.193476,0.193077,0.187741,0.190112,0.189115,0.191802,0.192679,0.182866,0


In [24]:
df_numeric_cols_REC94 = fill_missing_with_median_coding(Modulo1633_REC94_2024_fil[numeric_cols_REC94])
df_numeric_cols_REC94.info(verbose = True, show_counts = True)

Shape after transformation: (18335, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 19 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   IDX94_x  18335 non-null  float64
 1   BIDX     18335 non-null  float64
 2   ID1      18335 non-null  float64
 3   S410B    18335 non-null  float64
 4   S430C    18335 non-null  float64
 5   S431A    18335 non-null  float64
 6   S432     18335 non-null  float64
 7   S435     18335 non-null  float64
 8   S440     18335 non-null  float64
 9   S442     18335 non-null  float64
 10  S447     18335 non-null  float64
 11  QI411F   18335 non-null  float64
 12  QI440B   18335 non-null  float64
 13  S411BA   18335 non-null  float64
 14  S411DA   18335 non-null  float64
 15  S411CA   18335 non-null  float64
 16  S411EA   18335 non-null  float64
 17  S422I    18335 non-null  float64
 18  IDX94    18335 non-null  float64
dtypes: float64(19)
memory usage: 2.7 MB


In [25]:
df_load_dummy_cols_REC94 =  fill_missing_with_mode(Modulo1633_REC94_2024_fil[dummy_cols_REC94])
df_load_dummy_cols_REC94.info(verbose = True, show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 18 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   S413    18335 non-null  float64
 1   S426E   18335 non-null  float64
 2   S426GA  18335 non-null  float64
 3   S426GB  18335 non-null  float64
 4   S426GC  18335 non-null  float64
 5   S426GD  18335 non-null  float64
 6   S426GE  18335 non-null  float64
 7   S430D   18335 non-null  float64
 8   S427DA  18335 non-null  float64
 9   S427DB  18335 non-null  float64
 10  S427DC  18335 non-null  float64
 11  S427DD  18335 non-null  float64
 12  S427DE  18335 non-null  float64
 13  S427DF  18335 non-null  float64
 14  S427DG  18335 non-null  float64
 15  S427F   18335 non-null  float64
 16  S436C   18335 non-null  float64
 17  S441    18335 non-null  float64
dtypes: float64(18)
memory usage: 2.5 MB


In [26]:
df_categorical_cod_rec94.shape,  df_numeric_cols_REC94.shape, df_load_dummy_cols_REC94.shape

((18335, 17), (18335, 19), (18335, 18))

In [27]:
Modulo1633_REC94_2024_fil_clear = pd.concat([df_numeric_cols_REC94, 
                                             df_load_dummy_cols_REC94,
                                              df_categorical_cod_rec94], axis=1)

Modulo1633_REC94_2024_fil_clear.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 54 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   IDX94_x         18335 non-null  float64
 1   BIDX            18335 non-null  float64
 2   ID1             18335 non-null  float64
 3   S410B           18335 non-null  float64
 4   S430C           18335 non-null  float64
 5   S431A           18335 non-null  float64
 6   S432            18335 non-null  float64
 7   S435            18335 non-null  float64
 8   S440            18335 non-null  float64
 9   S442            18335 non-null  float64
 10  S447            18335 non-null  float64
 11  QI411F          18335 non-null  float64
 12  QI440B          18335 non-null  float64
 13  S411BA          18335 non-null  float64
 14  S411DA          18335 non-null  float64
 15  S411CA          18335 non-null  float64
 16  S411EA          18335 non-null  float64
 17  S422I           18335 non-null 

In [28]:
encoding_map_rec94

{'S411L': {'0.0': 0.2307582932620273,
  '1.0': 0.17827090632940568,
  '8.0': 0.351137384963986,
  'nan': 0.1944199928335902},
 'QI422A_C': {'1.0': 0.20055718123285565,
  '2.0': 0.2255095870447312,
  '8.0': 0.0,
  'nan': 0.1903329169951406},
 'QI422A_A': {'1.0': 0.19107895270772216,
  '2.0': 0.24261635765951195,
  '8.0': 0.15085811217129316,
  'nan': 0.19325961391567134},
 'S426FB': {'100.0': 0.18011182195357203,
  '101.0': 0.19999373713789612,
  '102.0': 0.2321171655030382,
  '103.0': 0.2237231504522903,
  '104.0': 0.14377474070789084,
  '105.0': 0.30194247964351323,
  '106.0': 0.0,
  '107.0': 0.0,
  '108.0': 0.08763371612481448,
  '110.0': 0.351137384963986,
  '113.0': 1.0,
  '115.0': 0.0,
  '116.0': 1.0,
  '120.0': 0.3631239723743611,
  '201.0': 0.0,
  '202.0': 0.0,
  '203.0': 1.0,
  '304.0': 1.0,
  '998.0': 0.3723038397927811,
  'nan': 0.21584250262532076},
 'QI422A_D': {'1.0': 0.2218307061642145,
  '2.0': 0.1769516264894459,
  'nan': 0.19061191001676256},
 'S411I': {'0.0': 0.239439

In [29]:
save_file(Modulo1633_REC94_2024_fil_clear, output_file = "Modulo1633_REC94_2024_fil_clear_v3.csv")
save_txt(encoding_map_rec94, dir="data\\interim",output_file="encoding_map_rec94.txt")


✅ File saved successfully at: c:\Users\linoc\OneDrive\Encoder\03_partos\02_scripts\Premature_model\data\interim\encoding_map_rec94.txt


## Modulo1631_REC91_2024

In [30]:
Modulo1631_REC91_2024.info(verbose= True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 343 columns):
 #    Column    Non-Null Count  Dtype 
---   ------    --------------  ----- 
 0    ID1       37117 non-null  int64 
 1    CASEID    37117 non-null  object
 2    SVER      37117 non-null  int64 
 3    SREGION   37117 non-null  int64 
 4    SSEMES    37117 non-null  int64 
 5    SPROVIN   37117 non-null  int64 
 6    SDISTRI   37117 non-null  int64 
 7    S108N     37117 non-null  object
 8    S108Y     37117 non-null  object
 9    S108G     37117 non-null  object
 10   S111      37117 non-null  object
 11   S112      37117 non-null  object
 12   S119      37117 non-null  object
 13   S119NA    37117 non-null  object
 14   S119NB    37117 non-null  object
 15   S119D     37117 non-null  object
 16   S229A     37117 non-null  object
 17   S229B     37117 non-null  object
 18   S229C     37117 non-null  object
 19   S229D     37117 non-null  object
 20   S229E     37117 non-null  

In [31]:
Modulo1631_REC91_2024_M = convert_objects_to_int64_safe(Modulo1631_REC91_2024)
Modulo1631_REC91_2024_M = Modulo1631_REC91_2024_M.dropna(axis=1, how="all")
Modulo1631_REC91_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 299 columns):
 #    Column    Non-Null Count  Dtype 
---   ------    --------------  ----- 
 0    ID1       37117 non-null  int64 
 1    CASEID    37117 non-null  object
 2    SVER      37117 non-null  int64 
 3    SREGION   37117 non-null  int64 
 4    SSEMES    37117 non-null  int64 
 5    SPROVIN   37117 non-null  int64 
 6    SDISTRI   37117 non-null  int64 
 7    S108N     34252 non-null  Int64 
 8    S108Y     33910 non-null  Int64 
 9    S108G     6326 non-null   Int64 
 10   S111      12058 non-null  Int64 
 11   S112      4623 non-null   Int64 
 12   S119      34252 non-null  Int64 
 13   S119NA    27810 non-null  Int64 
 14   S119NB    27810 non-null  Int64 
 15   S119D     34252 non-null  Int64 
 16   S229A     723 non-null    Int64 
 17   S229B     723 non-null    Int64 
 18   S229C     723 non-null    Int64 
 19   S229D     723 non-null    Int64 
 20   S229E     723 non-null    

In [32]:
Modulo1631_REC91_2024_M = drop_high_null_columns(Modulo1631_REC91_2024_M , threshold=0.45)
Modulo1631_REC91_2024_M.info(verbose = True, show_counts = True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 132 columns):
 #    Column   Non-Null Count  Dtype 
---   ------   --------------  ----- 
 0    ID1      37117 non-null  int64 
 1    CASEID   37117 non-null  object
 2    SVER     37117 non-null  int64 
 3    SREGION  37117 non-null  int64 
 4    SSEMES   37117 non-null  int64 
 5    SPROVIN  37117 non-null  int64 
 6    SDISTRI  37117 non-null  int64 
 7    S108N    34252 non-null  Int64 
 8    S108Y    33910 non-null  Int64 
 9    S119     34252 non-null  Int64 
 10   S119NA   27810 non-null  Int64 
 11   S119NB   27810 non-null  Int64 
 12   S119D    34252 non-null  Int64 
 13   S239A    34252 non-null  Int64 
 14   S239B    34252 non-null  Int64 
 15   S239C    34252 non-null  Int64 
 16   S239D    34252 non-null  Int64 
 17   S239E    34252 non-null  Int64 
 18   S239F    34252 non-null  Int64 
 19   S239X    34252 non-null  Int64 
 20   S489C    30623 non-null  Int64 
 21   S489D    3

In [33]:
categorical_cols_REC91 = ['SREGION',
                          'SPROVIN',
                          'SDISTRI',
                          'S108N',
                          'S112',
                          'S119',
                          'S119NA',
                          'S119NB',
                          'S119D',
                          'S229B1',
                          'S317AC',
                          'S317AD',
                          'S317C',
                          'S321A',
                          'S325A',
                          'S325B',
                          'S325D',
                          'S325E',
                          'S325GA',
                          'S325GB',
                          'S325GC',
                          'S325GD',
                          'S325GE',
                          'S325GF',
                          'S325GG',
                          'S325GH',
                          'S325GI', 
                          'S325GJ',
                          'S325GK',
                          'S621',
                          'S621A',
                          'S704N',
                          'S718',
                          'S720A',
                          'S802',
                          'S802D',
                          'S802E',
                          'S802F',
                          'S802H',
                          'S802I',
                          'S803A',
                          'S806AX',
                        'S806AZ',
                        'S807',
                        'S809',
                        'S8010',
                        'S801',
                        'S1002A',
                        'S1002B',
                        'S1002C',
                        'S1002D',
                        'S1002E',
                        'S1008AN',
                        'S1008BN',
                        'S1008CN',
                        'S1008DN',
                        'S1012BN',
                        'S1026',
                        'S1033',
                        'S1034A',
                        'S1034B',
                        'Q479A',
                        'Q479C'
                    ]

error = ['S112', 'S229B1', 'S317AC', 'S317AD', 'S317C', 'S321A', 'S325A', 'S325B', 'S325D', 'S325E', 'S325GA', 'S325GB', 'S325GC', 'S325GD', 'S325GE', 'S325GF', 'S325GG', 'S325GH', 'S325GI', 'S325GJ', 'S325GK', 'S621', 'S621A', 'S718', 'S720A', 'S802H', 'S803A', 'S8010', 'S801', 'S1008AN', 'S1008BN', 'S1008CN', 'S1008DN', 'S1012BN', 'Q479A', 'Q479C'] 

print(f'number total categor :{len(categorical_cols_REC91)}')
print(f'number Error :{len(error)}')

categorical_cols_REC91 = [col for col in categorical_cols_REC91 if col not in error]

print(f'number total categor after removal:{len(categorical_cols_REC91)}')

for i in Modulo1631_REC91_2024_M[categorical_cols_REC91].columns.to_list():
    print(Modulo1631_REC91_2024_M[i].value_counts())

number total categor :63
number Error :36
number total categor after removal:27
SREGION
3    11743
2    11200
4     9363
1     4811
Name: count, dtype: int64
SPROVIN
1     19006
3      3222
2      3023
6      2176
5      1921
7      1685
4      1422
8      1228
9      1171
11      630
18      484
10      464
12      355
13      141
19       64
17       58
20       27
15       17
16       15
14        8
Name: count, dtype: int64
SDISTRI
1     12479
2      2897
4      2758
5      2618
6      2367
3      2293
7      1790
10     1442
8      1296
9      1106
12      781
11      650
13      631
32      563
14      485
17      326
15      261
43      259
35      213
20      171
18      169
28      165
33      162
25      130
19      124
37      123
42      118
16      116
22       96
29       93
23       82
30       70
40       55
26       49
34       42
41       42
21       28
39       26
31       17
27       12
24       11
36        1
Name: count, dtype: int64
S108N
2    17124
1     6326
3 

In [35]:
numeric_cols_REC91, categorical_cols_REC91, dummy_cols_REC91 = categorize_columns (Modulo1631_REC91_2024_M, key_variables,categorical_cols_REC91)

🔹 Numeric columns: 9
🔹 Categorical columns: 27
🔹 Dummy columns: 95


In [36]:
categorical_cols_REC91.append('CASEID')

df_categorical_rec91 = Modulo1631_REC91_2024_M[categorical_cols_REC91].merge(target_final, how= 'left', on ='CASEID')

df_categorical_rec91 = fill_missing_with_mode(df_categorical_rec91)

# df_categorical_cod_rec91= target_encode_dataframe(df_categorical_rec91,target_col='premature_flag', 
#                             smoothing=0.3, min_samples_leaf=20)

df_categorical_cod_rec91, encoding_map_rec91, encoder_rec91 = target_encode_dataframe_map(df_categorical_rec91,
                                                               target_col='premature_flag', id_col='CASEID')

    
df_categorical_cod_rec91.head()

✅ Encoded 27 categorical columns.


,CASEID,S1002C,SPROVIN,S802,S1034B,S806AZ,S1033,SREGION,S1026,S119,...,S1034A,S809,S704N,S119NA,S806AX,S1002A,S119NB,S802F,S802E,premature_flag
0,325503101 2,0.090547,0.096994,0.097220,0.095056,0.097723,0.089674,0.091599,0.091974,0.095737,...,0.094139,0.098114,0.094692,0.095271,0.095539,0.086989,0.095400,0.099240,0.096206,1.0
1,325503101 4,0.091286,0.098470,0.097716,0.095003,0.087319,0.089877,0.089094,0.092348,0.095779,...,0.093862,0.098074,0.072367,0.095067,0.095677,0.086961,0.095464,0.085974,0.093605,0.0
2,325503901 2,0.089886,0.095768,0.077579,0.095012,0.097069,0.089406,0.090514,0.092623,0.095013,...,0.093637,0.084782,0.073294,0.095171,0.095641,0.086755,0.094913,0.099769,0.094825,0.0
3,325504701 2,0.118618,0.095768,0.097138,0.095012,0.078905,0.121753,0.090514,0.092623,0.095013,...,0.093637,0.084782,0.073294,0.095171,0.078905,0.125580,0.094913,0.099769,0.094825,0.0
4,325505001 1,0.091259,0.096855,0.097262,0.094880,0.098351,0.090038,0.093118,0.092579,0.095376,...,0.093958,0.098035,0.093315,0.094788,0.095425,0.087104,0.095143,0.098790,0.095714,0.0


In [37]:
df_numeric_cols_REC91 = fill_missing_with_median_coding(Modulo1631_REC91_2024_M[numeric_cols_REC91])
df_numeric_cols_REC91.info(verbose = True, show_counts = True)

Shape after transformation: (37117, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ID1     37117 non-null  float64
 1   SSEMES  37117 non-null  float64
 2   S108Y   37117 non-null  float64
 3   S489C   37117 non-null  float64
 4   S489D   37117 non-null  float64
 5   S490    37117 non-null  float64
 6   S704Y   37117 non-null  float64
 7   S810    37117 non-null  float64
 8   S811    37117 non-null  float64
dtypes: float64(9)
memory usage: 2.5 MB


In [38]:
df_load_dummy_cols_REC91 =  fill_missing_with_mode(Modulo1631_REC91_2024_M[dummy_cols_REC91])
df_load_dummy_cols_REC91.info(verbose = True, show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 95 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   SVER    37117 non-null  int64
 1   S239A   37117 non-null  Int64
 2   S239B   37117 non-null  Int64
 3   S239C   37117 non-null  Int64
 4   S239D   37117 non-null  Int64
 5   S239E   37117 non-null  Int64
 6   S239F   37117 non-null  Int64
 7   S239X   37117 non-null  Int64
 8   S490AA  37117 non-null  Int64
 9   S490AB  37117 non-null  Int64
 10  S490AC  37117 non-null  Int64
 11  S490AD  37117 non-null  Int64
 12  S490AE  37117 non-null  Int64
 13  S490AF  37117 non-null  Int64
 14  S490AG  37117 non-null  Int64
 15  S490AX  37117 non-null  Int64
 16  S490BA  37117 non-null  Int64
 17  S490BB  37117 non-null  Int64
 18  S490BC  37117 non-null  Int64
 19  S490BD  37117 non-null  Int64
 20  S490BX  37117 non-null  Int64
 21  S500A   37117 non-null  Int64
 22  S500B   37117 non-null  Int64
 23  S500C   371

In [39]:
df_categorical_cod_rec91.shape, df_load_dummy_cols_REC91.shape, df_numeric_cols_REC91.shape

((37117, 29), (37117, 95), (37117, 9))

In [41]:
Modulo1631_REC91_2024_M_clear = pd.concat([df_categorical_cod_rec91, 
                                           df_numeric_cols_REC91,
                                             df_load_dummy_cols_REC91], axis=1)
Modulo1631_REC91_2024_M_clear.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37117 entries, 0 to 37116
Data columns (total 133 columns):
 #    Column          Non-Null Count  Dtype  
---   ------          --------------  -----  
 0    CASEID          37117 non-null  object 
 1    S1002C          37117 non-null  float64
 2    SPROVIN         37117 non-null  float64
 3    S802            37117 non-null  float64
 4    S1034B          37117 non-null  float64
 5    S806AZ          37117 non-null  float64
 6    S1033           37117 non-null  float64
 7    SREGION         37117 non-null  float64
 8    S1026           37117 non-null  float64
 9    S119            37117 non-null  float64
 10   S119D           37117 non-null  float64
 11   S807            37117 non-null  float64
 12   S802D           37117 non-null  float64
 13   S1002E          37117 non-null  float64
 14   SDISTRI         37117 non-null  float64
 15   S802I           37117 non-null  float64
 16   S108N           37117 non-null  float64
 17   S1002B    

In [42]:
save_file(Modulo1631_REC91_2024_M_clear, output_file = "Modulo1631_REC91_2024_M_clear_v3.csv")
save_txt(encoding_map_rec91, dir="data\\interim",output_file="encoding_map_rec91.txt")

✅ File saved successfully at: c:\Users\linoc\OneDrive\Encoder\03_partos\02_scripts\Premature_model\data\interim\encoding_map_rec91.txt


## Modulo1632_RE223132_2024

In [43]:
Modulo1632_RE223132_2024.info(verbose = True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 149 columns):
 #    Column     Non-Null Count  Dtype 
---   ------     --------------  ----- 
 0    ID1        34252 non-null  int64 
 1    CASEID     34252 non-null  object
 2    V201       34252 non-null  int64 
 3    V202       34252 non-null  int64 
 4    V203       34252 non-null  int64 
 5    V204       34252 non-null  int64 
 6    V205       34252 non-null  int64 
 7    V206       34252 non-null  int64 
 8    V207       34252 non-null  int64 
 9    V208       34252 non-null  int64 
 10   V209       34252 non-null  int64 
 11   V210       34252 non-null  int64 
 12   V211       34252 non-null  object
 13   V212       34252 non-null  object
 14   V213       34252 non-null  int64 
 15   V214       34252 non-null  object
 16   V215       34252 non-null  int64 
 17   V216       34252 non-null  int64 
 18   V217       34252 non-null  int64 
 19   V218       34252 non-null  int64 
 20   V219

In [44]:
Modulo1632_RE223132_2024_M = convert_objects_to_int64_safe(Modulo1632_RE223132_2024)
Modulo1632_RE223132_2024_M = Modulo1632_RE223132_2024_M.dropna(axis=1, how="all")
Modulo1632_RE223132_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 135 columns):
 #    Column     Non-Null Count  Dtype 
---   ------     --------------  ----- 
 0    ID1        34252 non-null  int64 
 1    CASEID     34252 non-null  object
 2    V201       34252 non-null  int64 
 3    V202       34252 non-null  int64 
 4    V203       34252 non-null  int64 
 5    V204       34252 non-null  int64 
 6    V205       34252 non-null  int64 
 7    V206       34252 non-null  int64 
 8    V207       34252 non-null  int64 
 9    V208       34252 non-null  int64 
 10   V209       34252 non-null  int64 
 11   V210       34252 non-null  int64 
 12   V211       24627 non-null  Int64 
 13   V212       24627 non-null  Int64 
 14   V213       34252 non-null  int64 
 15   V214       723 non-null    Int64 
 16   V215       34252 non-null  int64 
 17   V216       34252 non-null  int64 
 18   V217       34252 non-null  int64 
 19   V218       34252 non-null  int64 
 20   V219

In [45]:
Modulo1632_RE223132_2024_M = drop_high_null_columns(Modulo1632_RE223132_2024_M , threshold=0.45)
Modulo1632_RE223132_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 65 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID1        34252 non-null  int64 
 1   CASEID     34252 non-null  object
 2   V201       34252 non-null  int64 
 3   V202       34252 non-null  int64 
 4   V203       34252 non-null  int64 
 5   V204       34252 non-null  int64 
 6   V205       34252 non-null  int64 
 7   V206       34252 non-null  int64 
 8   V207       34252 non-null  int64 
 9   V208       34252 non-null  int64 
 10  V209       34252 non-null  int64 
 11  V210       34252 non-null  int64 
 12  V211       24627 non-null  Int64 
 13  V212       24627 non-null  Int64 
 14  V213       34252 non-null  int64 
 15  V215       34252 non-null  int64 
 16  V216       34252 non-null  int64 
 17  V217       34252 non-null  int64 
 18  V218       34252 non-null  int64 
 19  V219       34252 non-null  int64 
 20  V220       34252 non-null  i

In [46]:

categorical_cols_RE223132 = ['V312',
                          'V313',
                        #   'V326',
                        #   'V327',
                          'V359',
                          'V360',
                          'V361',
                        #   'V362',
                        #   'V363',
                          'V364',
                        #   'V367',
                        #   'V375A',
                        #   'V376',
                        #   'V376A',
                        #   'V379',
                        #   'V380',
                        #   'V3A07'
                    ]

# error = ['S112', 'S229B1', 'S317AC', 'S317AD', 'S317C', 'S321A', 'S325A', 'S325B', 'S325D', 'S325E', 'S325GA', 'S325GB', 'S325GC', 'S325GD', 'S325GE', 'S325GF', 'S325GG', 'S325GH', 'S325GI', 'S325GJ', 'S325GK', 'S621', 'S621A', 'S718', 'S720A', 'S802H', 'S803A', 'S8010', 'S801', 'S1008AN', 'S1008BN', 'S1008CN', 'S1008DN', 'S1012BN', 'Q479A', 'Q479C'] 

# print(f'number total categor :{len(categorical_cols_REC91)}')
# print(f'number Error :{len(error)}')

# categorical_cols_REC91 = [col for col in categorical_cols_REC91 if col not in error]

# print(f'number total categor after removal:{len(categorical_cols_REC91)}')

for i in Modulo1632_RE223132_2024_M[categorical_cols_RE223132].columns.to_list():
    print(Modulo1632_RE223132_2024_M[i].value_counts())




V312
0     14652
3      5941
5      3029
6      2552
11     2546
8      2001
9      1773
1      1274
2       229
10      141
7        52
16       43
13       14
15        5
Name: count, dtype: int64
V313
3    15685
0    14652
2     3774
1      141
Name: count, dtype: int64
V359
3     6199
5     4510
9     2769
8     2613
1     1941
11    1011
16     573
10     252
2      126
13      50
15      16
6        5
14       5
7        3
Name: count, dtype: Int64
V360
1     3903
2     3774
4     3404
13    2229
7     1885
9     1652
8     1040
14     876
6      455
5      272
12     232
3      171
11     154
10      24
98       2
Name: count, dtype: Int64
V361
1    19600
4     7028
2     5135
3     2489
Name: count, dtype: int64
V364
1    15685
3    12217
2     3915
4     2435
Name: count, dtype: int64


In [47]:
numeric_cols_RE223132, categorical_cols_RE223132, dummy_cols_RE223132 = categorize_columns (Modulo1632_RE223132_2024_M, key_variables,categorical_cols_RE223132)

🔹 Numeric columns: 35
🔹 Categorical columns: 6
🔹 Dummy columns: 23


In [48]:
categorical_cols_RE223132.append('CASEID')

df_categorical_RE223132 = Modulo1632_RE223132_2024_M[categorical_cols_RE223132].merge(target_final, how= 'left', on ='CASEID')

df_categorical_RE223132  = fill_missing_with_mode(df_categorical_RE223132)

# df_categorical_RE223132= target_encode_dataframe(df_categorical_RE223132,target_col='premature_flag', 
#                             smoothing=0.3, min_samples_leaf=20)

df_categorical_cod_RE223132, encoding_map_RE223132, encoder_RE223132 = target_encode_dataframe_map(df_categorical_RE223132,
                                                               target_col='premature_flag', id_col='CASEID')
    
df_categorical_RE223132.head()

✅ Encoded 6 categorical columns.


,V313,V364,V359,V360,V312,V361,CASEID,premature_flag
0,2,2,3,1,8,1,325503101 2,1.0
1,0,3,3,1,0,4,325503101 4,0.0
2,0,3,3,1,0,4,325503901 2,0.0
3,2,2,3,1,9,1,325504701 2,0.0
4,3,1,3,4,1,1,325505001 1,0.0


In [49]:
df_categorical_cod_RE223132.head()

,CASEID,V313,V364,V359,V360,V312,V361,premature_flag
0,325503101 2,0.107153,0.107829,0.078865,0.074687,0.104731,0.143155,1.0
1,325503101 4,0.047515,0.047769,0.077752,0.077143,0.047515,0.003588,0.0
2,325503901 2,0.049776,0.051169,0.078865,0.074687,0.049776,0.004937,0.0
3,325504701 2,0.107153,0.107829,0.078865,0.074687,0.109854,0.143155,0.0
4,325505001 1,0.153347,0.153347,0.077752,0.137253,0.112265,0.144517,0.0


In [50]:
df_numeric_cols_RE223132 = fill_missing_with_median_coding(Modulo1632_RE223132_2024_M[numeric_cols_RE223132])
df_numeric_cols_RE223132.info(verbose = True, show_counts = True)

Shape after transformation: (34252, 35)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 35 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID1        34252 non-null  float64
 1   V201       34252 non-null  float64
 2   V202       34252 non-null  float64
 3   V203       34252 non-null  float64
 4   V204       34252 non-null  float64
 5   V205       34252 non-null  float64
 6   V206       34252 non-null  float64
 7   V207       34252 non-null  float64
 8   V208       34252 non-null  float64
 9   V209       34252 non-null  float64
 10  V210       34252 non-null  float64
 11  V211       34252 non-null  float64
 12  V212       34252 non-null  float64
 13  V215       34252 non-null  float64
 14  V217       34252 non-null  float64
 15  V218       34252 non-null  float64
 16  V219       34252 non-null  float64
 17  V220       34252 non-null  float64
 18  V221       34252 non-null  float64
 19  V222  

In [51]:
df_load_dummy_cols_RE223132 =  fill_missing_with_mode(Modulo1632_RE223132_2024_M[dummy_cols_RE223132])
df_load_dummy_cols_RE223132.info(verbose = True, show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 23 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   V213     34252 non-null  int64
 1   V216     34252 non-null  int64
 2   V228     34252 non-null  int64
 3   V384A    34252 non-null  int64
 4   V384B    34252 non-null  int64
 5   V384C    34252 non-null  int64
 6   V393     34252 non-null  Int64
 7   V394     34252 non-null  Int64
 8   V305_01  34252 non-null  int64
 9   V305_02  34252 non-null  int64
 10  V305_03  34252 non-null  int64
 11  V305_05  34252 non-null  int64
 12  V305_06  34252 non-null  int64
 13  V305_07  34252 non-null  int64
 14  V305_08  34252 non-null  int64
 15  V305_09  34252 non-null  int64
 16  V305_10  34252 non-null  int64
 17  V305_11  34252 non-null  int64
 18  V305_13  34252 non-null  int64
 19  V305_14  34252 non-null  int64
 20  V305_15  34252 non-null  int64
 21  V305_16  34252 non-null  int64
 22  V307_03  34252 non-nul

In [52]:
df_numeric_cols_RE223132.shape, df_load_dummy_cols_RE223132.shape, 

((34252, 35), (34252, 23))

In [53]:
Modulo1632_RE223132_2024_M_clear = pd.concat([df_numeric_cols_RE223132,
                                               df_load_dummy_cols_RE223132,
                                                 df_categorical_cod_RE223132], axis=1)
Modulo1632_RE223132_2024_M_clear.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 66 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID1             34252 non-null  float64
 1   V201            34252 non-null  float64
 2   V202            34252 non-null  float64
 3   V203            34252 non-null  float64
 4   V204            34252 non-null  float64
 5   V205            34252 non-null  float64
 6   V206            34252 non-null  float64
 7   V207            34252 non-null  float64
 8   V208            34252 non-null  float64
 9   V209            34252 non-null  float64
 10  V210            34252 non-null  float64
 11  V211            34252 non-null  float64
 12  V212            34252 non-null  float64
 13  V215            34252 non-null  float64
 14  V217            34252 non-null  float64
 15  V218            34252 non-null  float64
 16  V219            34252 non-null  float64
 17  V220            34252 non-null 

In [54]:
save_file(Modulo1632_RE223132_2024_M_clear, output_file = "Modulo1632_RE223132_2024_M_clear_v3.csv")
save_txt(encoding_map_RE223132, dir="data\\interim",output_file="encoding_map_RE223132.txt")

✅ File saved successfully at: c:\Users\linoc\OneDrive\Encoder\03_partos\02_scripts\Premature_model\data\interim\encoding_map_RE223132.txt


## Modulo1635_RE516171_2024

In [55]:
Modulo1635_RE516171_2024.info(verbose= True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 84 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID1     34252 non-null  int64 
 1   CASEID  34252 non-null  object
 2   V501    34252 non-null  int64 
 3   V502    34252 non-null  int64 
 4   V503    34252 non-null  object
 5   V504    34252 non-null  object
 6   V505    34252 non-null  object
 7   V506    34252 non-null  object
 8   V507    34252 non-null  object
 9   V508    34252 non-null  object
 10  V509    34252 non-null  object
 11  V510    34252 non-null  object
 12  V511    34252 non-null  object
 13  V512    34252 non-null  object
 14  V513    34252 non-null  int64 
 15  V525    34252 non-null  int64 
 16  V527    34252 non-null  object
 17  V528    34252 non-null  object
 18  V529    34252 non-null  object
 19  V530    34252 non-null  object
 20  V531    34252 non-null  int64 
 21  V532    34252 non-null  int64 
 22  V535    34252 non-null

In [56]:
Modulo1635_RE516171_2024_M = convert_objects_to_int64_safe(Modulo1635_RE516171_2024)
Modulo1635_RE516171_2024_M = Modulo1635_RE516171_2024_M.dropna(axis=1, how="all")
Modulo1635_RE516171_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 76 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID1     34252 non-null  int64 
 1   CASEID  34252 non-null  object
 2   V501    34252 non-null  int64 
 3   V502    34252 non-null  int64 
 4   V503    24189 non-null  Int64 
 5   V504    19780 non-null  Int64 
 6   V507    24189 non-null  Int64 
 7   V508    24189 non-null  Int64 
 8   V509    24189 non-null  Int64 
 9   V510    24189 non-null  Int64 
 10  V511    24189 non-null  Int64 
 11  V512    24189 non-null  Int64 
 12  V513    34252 non-null  int64 
 13  V525    34252 non-null  int64 
 14  V527    27705 non-null  Int64 
 15  V528    27705 non-null  Int64 
 16  V529    27705 non-null  Int64 
 17  V530    27705 non-null  Int64 
 18  V531    34252 non-null  int64 
 19  V532    34252 non-null  int64 
 20  V535    14472 non-null  Int64 
 21  V536    34252 non-null  int64 
 22  V537    8581 non-null 

In [57]:
Modulo1635_RE516171_2024_M = drop_high_null_columns(Modulo1635_RE516171_2024_M , threshold=0.45)
Modulo1635_RE516171_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 63 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID1     34252 non-null  int64 
 1   CASEID  34252 non-null  object
 2   V501    34252 non-null  int64 
 3   V502    34252 non-null  int64 
 4   V503    24189 non-null  Int64 
 5   V504    19780 non-null  Int64 
 6   V507    24189 non-null  Int64 
 7   V508    24189 non-null  Int64 
 8   V509    24189 non-null  Int64 
 9   V510    24189 non-null  Int64 
 10  V511    24189 non-null  Int64 
 11  V512    24189 non-null  Int64 
 12  V513    34252 non-null  int64 
 13  V525    34252 non-null  int64 
 14  V527    27705 non-null  Int64 
 15  V528    27705 non-null  Int64 
 16  V529    27705 non-null  Int64 
 17  V530    27705 non-null  Int64 
 18  V531    34252 non-null  int64 
 19  V532    34252 non-null  int64 
 20  V536    34252 non-null  int64 
 21  V602    34252 non-null  int64 
 22  V605    34252 non-null

In [58]:
categorical_cols_RE516171 = ['V501','V502','V503',
'V504',
'V510',
# 'V535',
'V536',
# 'V539',
'V602',
'V605',
'V631',
# 'V632',
'V633A',
'V633B', 
'V633C',
'V633D',
# 'V633E',
# 'V633F',
# 'V633G',
# 'V634',
'V701',
'V704',
'V705',
'V716',
'V717',
'V719',
'V729',
'V731',
'V732',
# 'V739',
'V743A','V743B','V743C','V743D','V743E','V743F','V744A','V744B','V744C','V744D','V744E',
# 'V746'
]



# error = ['S112', 'S229B1', 'S317AC', 'S317AD', 'S317C', 'S321A', 'S325A', 'S325B', 'S325D', 'S325E', 'S325GA', 'S325GB', 'S325GC', 'S325GD', 'S325GE', 'S325GF', 'S325GG', 'S325GH', 'S325GI', 'S325GJ', 'S325GK', 'S621', 'S621A', 'S718', 'S720A', 'S802H', 'S803A', 'S8010', 'S801', 'S1008AN', 'S1008BN', 'S1008CN', 'S1008DN', 'S1012BN', 'Q479A', 'Q479C'] 

# print(f'number total categor :{len(categorical_cols_REC91)}')
# print(f'number Error :{len(error)}')

# categorical_cols_REC91 = [col for col in categorical_cols_REC91 if col not in error]

# print(f'number total categor after removal:{len(categorical_cols_REC91)}')

for i in Modulo1635_RE516171_2024_M[categorical_cols_RE516171].columns.to_list():
    print(Modulo1635_RE516171_2024_M[i].value_counts())



V501
2    14819
0    10063
1     4961
5     4251
4       83
3       75
Name: count, dtype: int64
V502
1    19780
0    10063
2     4409
Name: count, dtype: int64
V503
1    19775
2     4414
Name: count, dtype: Int64
V504
1    18088
2     1692
Name: count, dtype: Int64
V510
1    22985
5     1171
6       33
Name: count, dtype: Int64
V536
1    19124
3     6788
0     6547
2     1793
Name: count, dtype: int64
V602
3    16372
1    14638
4     2604
5      470
2      168
Name: count, dtype: int64
V605
5    16372
2    12684
6     2604
1     1579
7      470
3      375
4      168
Name: count, dtype: int64
V631
1    15284
3     8483
2     4904
4      440
Name: count, dtype: Int64
V633A
1    32996
0      858
8      398
Name: count, dtype: int64
V633B
1    33318
0      646
8      288
Name: count, dtype: int64
V633C
1    33272
0      687
8      293
Name: count, dtype: int64
V633D
1    32839
0     1050
8      363
Name: count, dtype: int64
V701
2    12442
3     8064
1     3469
0      144
8       70
Name:

In [62]:
numeric_cols_RE516171, categorical_cols_RE516171, dummy_cols_RE516171 = categorize_columns (Modulo1635_RE516171_2024_M, key_variables,categorical_cols_RE516171)

🔹 Numeric columns: 28
🔹 Categorical columns: 33
🔹 Dummy columns: 1


In [63]:
categorical_cols_RE516171.append('CASEID')

df_categorical_RE516171 = Modulo1635_RE516171_2024_M[categorical_cols_RE516171].merge(target_final, how= 'left', on ='CASEID')

df_categorical_RE516171  = fill_missing_with_mode(df_categorical_RE516171)

# df_categorical_RE516171= target_encode_dataframe(df_categorical_RE516171,target_col='premature_flag', 
#                             smoothing=0.3, min_samples_leaf=20)


df_categorical_cod_RE516171, encoding_map_RE516171, encoder_RE516171 = target_encode_dataframe_map(df_categorical_RE516171,
                                                               target_col='premature_flag', id_col='CASEID')
    
df_categorical_RE516171.head()

✅ Encoded 33 categorical columns.


,V743D,V743E,V510,V602,V744A,V743A,V732,V633C,V536,V633B,...,V705,V744B,V743B,V605,V504,V719,V502,V633D,CASEID,premature_flag
0,2,1,1,3,0,1,1,1,1,1,...,8,0,2,5,1,2,1,1,325503101 2,1.0
1,2,1,1,1,0,1,1,8,0,1,...,4,0,2,2,1,2,0,8,325503101 4,0.0
2,2,1,1,1,0,1,1,1,0,1,...,4,0,2,2,1,2,0,1,325503901 2,0.0
3,2,2,1,3,0,1,1,1,1,1,...,6,0,2,5,1,2,1,1,325504701 2,0.0
4,2,1,1,1,0,2,1,1,1,1,...,8,0,2,2,1,2,1,1,325505001 1,0.0


In [64]:
df_numeric_cols_RE516171 = fill_missing_with_median_coding(Modulo1635_RE516171_2024_M[numeric_cols_RE516171])
df_numeric_cols_RE516171.info(verbose = True, show_counts = True)

Shape after transformation: (34252, 28)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 28 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ID1     34252 non-null  float64
 1   V507    34252 non-null  float64
 2   V508    34252 non-null  float64
 3   V509    34252 non-null  float64
 4   V511    34252 non-null  float64
 5   V512    34252 non-null  float64
 6   V513    34252 non-null  float64
 7   V525    34252 non-null  float64
 8   V527    34252 non-null  float64
 9   V528    34252 non-null  float64
 10  V529    34252 non-null  float64
 11  V530    34252 non-null  float64
 12  V531    34252 non-null  float64
 13  V532    34252 non-null  float64
 14  V613    34252 non-null  float64
 15  V614    34252 non-null  float64
 16  V623    34252 non-null  float64
 17  V624    34252 non-null  float64
 18  V625    34252 non-null  float64
 19  V626    34252 non-null  float64
 20  V627    34252 non-null  float64


In [65]:
df_load_dummy_cols_RE516171 =  fill_missing_with_mode(Modulo1635_RE516171_2024_M[dummy_cols_RE516171])
df_load_dummy_cols_RE516171.info(verbose = True, show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   V714    34252 non-null  int64
dtypes: int64(1)
memory usage: 267.7 KB


In [66]:
Modulo1635_RE516171_2024_M_clear = pd.concat([df_numeric_cols_RE516171, 
                                              df_load_dummy_cols_RE516171,
                                                df_categorical_cod_RE516171], axis=1)
Modulo1635_RE516171_2024_M_clear.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34252 entries, 0 to 34251
Data columns (total 64 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID1             34252 non-null  float64
 1   V507            34252 non-null  float64
 2   V508            34252 non-null  float64
 3   V509            34252 non-null  float64
 4   V511            34252 non-null  float64
 5   V512            34252 non-null  float64
 6   V513            34252 non-null  float64
 7   V525            34252 non-null  float64
 8   V527            34252 non-null  float64
 9   V528            34252 non-null  float64
 10  V529            34252 non-null  float64
 11  V530            34252 non-null  float64
 12  V531            34252 non-null  float64
 13  V532            34252 non-null  float64
 14  V613            34252 non-null  float64
 15  V614            34252 non-null  float64
 16  V623            34252 non-null  float64
 17  V624            34252 non-null 

In [67]:
encoding_map_RE516171

{'V743D': {'0': 0.1203633594112511,
  '1': 0.1457636133024487,
  '2': 0.09629782310134083,
  '3': 0.0969635167163105,
  '4': 0.11574073414449072,
  '5': 0.11611203135651278},
 'V743E': {'0': 0.1001473142056087,
  '1': 0.09355102620073219,
  '2': 0.16252000516341472,
  '3': 0.11161388753945603,
  '4': 0.14486037760713413,
  '5': 0.18433560495130968},
 'V510': {'1': 0.10245705382957729,
  '5': 0.121246106711773,
  '6': 0.17816492609978926},
 'V602': {'1': 0.06893180433230499,
  '2': 0.05968044533702935,
  '3': 0.11427973699224073,
  '4': 0.23876098301535453,
  '5': 0.0468662811737324},
 'V744A': {'0': 0.10310551908289024, '1': 0.1206035041484911, '8': 0.0},
 'V743A': {'0': 0.2419802462791059,
  '1': 0.09833040685696708,
  '2': 0.12964322973686435,
  '3': 0.0570099130018505,
  '4': 0.11117081645419798,
  '5': 0.1923549879480032},
 'V732': {'1': 0.09866252651072997,
  '2': 0.114104740380939,
  '3': 0.1135892274315984},
 'V633C': {'0': 0.08590197380766862,
  '1': 0.10417164571792414,
  '8':

In [68]:
save_file(Modulo1635_RE516171_2024_M_clear, output_file = "Modulo1635_RE516171_2024_M_clear_v3.csv")

save_txt(encoding_map_RE516171, dir="data\\interim",output_file="encoding_map_RE516171.txt")

✅ File saved successfully at: c:\Users\linoc\OneDrive\Encoder\03_partos\02_scripts\Premature_model\data\interim\encoding_map_RE516171.txt


## Modulo1640_CSALUD01_2024


In [18]:
Modulo1640_CSALUD01_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1640/968-Modulo1640/CSALUD01_2024.csv", low_memory=False)
Modulo1640_CSALUD01_2024.info(verbose= True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34018 entries, 0 to 34017
Data columns (total 258 columns):
 #    Column       Non-Null Count  Dtype 
---   ------       --------------  ----- 
 0    ID1          34018 non-null  int64 
 1    HHID         34018 non-null  int64 
 2    QHCLUSTER    34018 non-null  int64 
 3    QHNUMBER     34018 non-null  int64 
 4    QHHOME       34018 non-null  int64 
 5    QSNUMERO     34018 non-null  int64 
 6    QSINTM       34018 non-null  int64 
 7    QSINTY       34018 non-null  int64 
 8    QSTOTVISIT   34018 non-null  int64 
 9    QSRESULT     34018 non-null  int64 
 10   QSNINOS      34018 non-null  object
 11   QSRESINF     34018 non-null  int64 
 12   QS20C        34018 non-null  object
 13   QSSEXO       34018 non-null  object
 14   QSMEF        34018 non-null  object
 15   QSDIA        34018 non-null  object
 16   QS22M        34018 non-null  object
 17   QS22A        34018 non-null  object
 18   QS23         34018 non-null  object
 19   QS

In [70]:
Modulo1640_CSALUD01_2024.shape, Modulo1640_CSALUD01_2024.HHID.nunique()

((34018, 258), 34018)

In [71]:
Modulo1640_CSALUD01_2024_M = convert_objects_to_int64_safe(Modulo1640_CSALUD01_2024)
Modulo1640_CSALUD01_2024_M = Modulo1640_CSALUD01_2024_M.dropna(axis=1, how="all")
Modulo1640_CSALUD01_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34018 entries, 0 to 34017
Data columns (total 243 columns):
 #    Column       Non-Null Count  Dtype 
---   ------       --------------  ----- 
 0    ID1          34018 non-null  int64 
 1    HHID         34018 non-null  int64 
 2    QHCLUSTER    34018 non-null  int64 
 3    QHNUMBER     34018 non-null  int64 
 4    QHHOME       34018 non-null  int64 
 5    QSNUMERO     34018 non-null  int64 
 6    QSINTM       34018 non-null  int64 
 7    QSINTY       34018 non-null  int64 
 8    QSTOTVISIT   34018 non-null  int64 
 9    QSRESULT     34018 non-null  int64 
 10   QSRESINF     34018 non-null  int64 
 11   QS20C        31487 non-null  Int64 
 12   QSSEXO       31487 non-null  Int64 
 13   QSMEF        31487 non-null  Int64 
 14   QSDIA        31484 non-null  Int64 
 15   QS22M        31484 non-null  Int64 
 16   QS22A        31484 non-null  Int64 
 17   QS23         31484 non-null  Int64 
 18   QS24         31484 non-null  Int64 
 19   QS

In [72]:
Modulo1640_CSALUD01_2024_M = drop_high_null_columns(Modulo1640_CSALUD01_2024_M , threshold=0.45)
Modulo1640_CSALUD01_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34018 entries, 0 to 34017
Data columns (total 94 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID1          34018 non-null  int64 
 1   HHID         34018 non-null  int64 
 2   QHCLUSTER    34018 non-null  int64 
 3   QHNUMBER     34018 non-null  int64 
 4   QHHOME       34018 non-null  int64 
 5   QSNUMERO     34018 non-null  int64 
 6   QSINTM       34018 non-null  int64 
 7   QSINTY       34018 non-null  int64 
 8   QSTOTVISIT   34018 non-null  int64 
 9   QSRESULT     34018 non-null  int64 
 10  QSRESINF     34018 non-null  int64 
 11  QS20C        31487 non-null  Int64 
 12  QSSEXO       31487 non-null  Int64 
 13  QSMEF        31487 non-null  Int64 
 14  QSDIA        31484 non-null  Int64 
 15  QS22M        31484 non-null  Int64 
 16  QS22A        31484 non-null  Int64 
 17  QS23         31484 non-null  Int64 
 18  QS24         31484 non-null  Int64 
 19  QS25N        30374 non-nu

In [73]:
# QS25AA,QS25BB,QS25C1,QS25C2,QS25C3,
# QS25C4,QS25C5,QS25C6,QS26,QS27,
# QS29A,QS29B,QS100,QS101,QS102,
# QS109,QS111,QS200,QS201,QS202,
# QS206,QS208,QS209,QS210,QS212T,
# QS212A,
# QS212B,
# QS212C,
# QS212D,
# QS700AQS212E,QS212F,QS212G,QS303,
# QS313,QS407,QS409,QS411,QS700A,QS700B,
# QS700AC,QS700AD,QS700AE,QS700AF,
# QS700AG,QS700AH,QS700AI,QS702,QS703,QS704A,QS704B, QS704C,QS704D, QS704E, QS704F,
#  QS704G, QS704H, QS704I,QS706,QS707,QS709,QS710,QS711,


# Ensure CASEID is string
target_final["CASEID_9d"] = (
    target_final["CASEID"]
    .astype(str)     # convert to string
    .str.strip()     # remove extra spaces
    .str[:9]         # take first 9 characters
)

# If you want numeric again (preserves the 9 digits correctly)
target_final["CASEID_9d"] = (
    target_final["CASEID_9d"].astype(int))

target_final = target_final[~target_final['CASEID_9d'].duplicated(keep=False)]

Modulo1640_CSALUD01_2024_M = Modulo1640_CSALUD01_2024_M.merge(target_final, right_on= 'CASEID_9d', left_on  = 'HHID', how = 'left')
Modulo1640_CSALUD01_2024_M.info(verbose=True, show_counts=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34018 entries, 0 to 34017
Data columns (total 97 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID1             34018 non-null  int64  
 1   HHID            34018 non-null  int64  
 2   QHCLUSTER       34018 non-null  int64  
 3   QHNUMBER        34018 non-null  int64  
 4   QHHOME          34018 non-null  int64  
 5   QSNUMERO        34018 non-null  int64  
 6   QSINTM          34018 non-null  int64  
 7   QSINTY          34018 non-null  int64  
 8   QSTOTVISIT      34018 non-null  int64  
 9   QSRESULT        34018 non-null  int64  
 10  QSRESINF        34018 non-null  int64  
 11  QS20C           31487 non-null  Int64  
 12  QSSEXO          31487 non-null  Int64  
 13  QSMEF           31487 non-null  Int64  
 14  QSDIA           31484 non-null  Int64  
 15  QS22M           31484 non-null  Int64  
 16  QS22A           31484 non-null  Int64  
 17  QS23            31484 non-null 

In [74]:
target_final.shape, target_final.CASEID_9d.nunique(),Modulo1640_CSALUD01_2024_M.shape,Modulo1640_CSALUD01_2024_M.HHID.nunique()

((17548, 3), 17548, (34018, 97), 34018)

In [75]:
df_categorical_1640_CSALUD01  = fill_missing_with_mode(Modulo1640_CSALUD01_2024_M)

# df_categorical_CSALUD01_cod= target_encode_dataframe(df_categorical_1640_CSALUD01,target_col='premature_flag', 
#                             smoothing=0.3, min_samples_leaf=20)


df_categorical_CSALUD01_cod, encoding_map_CSALUD01, encoder_CSALUD01 = target_encode_dataframe_map(df_categorical_1640_CSALUD01,
                                                               target_col='premature_flag', id_col='CASEID')
    
df_categorical_CSALUD01_cod.head()

✅ Encoded 95 categorical columns.


,CASEID,ID1,HHID,QHCLUSTER,QHNUMBER,QHHOME,QSNUMERO,QSINTM,QSINTY,QSTOTVISIT,...,QS25C2,QS25C3,QS25C4,QS25C5,QS25C6,QS907,QS908,PESO15_AMAS,CASEID_9d,premature_flag
0,325503101 2,0.099614,0.099614,0.000000,0.089878,0.099428,0.106197,0.106833,0.099614,0.082160,...,0.099720,0.099551,0.099786,0.099599,0.099614,0.174104,0.099186,0.099614,0.000000,1.0
1,325503101 2,0.099614,0.099614,0.135800,0.101314,0.099283,0.108374,0.098561,0.099614,0.082953,...,0.099771,0.099548,0.099760,0.099507,0.099595,0.227688,0.099209,0.099614,0.000076,0.0
2,325503101 2,0.099581,0.099581,0.135793,0.086320,0.099317,0.089479,0.088385,0.099581,0.083762,...,0.099757,0.099595,0.099779,0.099562,0.099702,0.063592,0.099020,0.099581,0.000076,0.0
3,325504701 2,0.099581,0.099581,0.135793,0.116297,0.099317,0.089479,0.088385,0.099581,0.083762,...,0.099757,0.099595,0.099779,0.099562,0.099702,0.101991,0.099020,0.000000,0.099581,0.0
4,325505001 1,0.099581,0.099581,0.152901,0.107403,0.099324,0.109245,0.092258,0.099581,0.132320,...,0.099668,0.099529,0.099771,0.099462,0.099591,0.083561,0.099315,0.000000,0.099581,0.0


In [76]:
df_categorical_CSALUD01_cod.drop(columns=['HHID'], inplace = True)

Modulo1640_CSALUD01_2024_M_clear_v3 = pd.concat([df_categorical_CSALUD01_cod, df_categorical_1640_CSALUD01[['HHID']], ], axis=1)
Modulo1640_CSALUD01_2024_M_clear_v3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34018 entries, 0 to 34017
Data columns (total 97 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CASEID          34018 non-null  object 
 1   ID1             34018 non-null  float64
 2   QHCLUSTER       34018 non-null  float64
 3   QHNUMBER        34018 non-null  float64
 4   QHHOME          34018 non-null  float64
 5   QSNUMERO        34018 non-null  float64
 6   QSINTM          34018 non-null  float64
 7   QSINTY          34018 non-null  float64
 8   QSTOTVISIT      34018 non-null  float64
 9   QSRESULT        34018 non-null  float64
 10  QSRESINF        34018 non-null  float64
 11  QS20C           34018 non-null  float64
 12  QSSEXO          34018 non-null  float64
 13  QSMEF           34018 non-null  float64
 14  QSDIA           34018 non-null  float64
 15  QS22M           34018 non-null  float64
 16  QS22A           34018 non-null  float64
 17  QS23            34018 non-null 

In [ ]:
save_file(Modulo1640_CSALUD01_2024_M_clear_v3, output_file = "Modulo1640_CSALUD01_2024_M_clear_v3.csv")

save_txt(encoding_map_CSALUD01, dir="data\\interim",output_file="encoding_map_CSALUD01.txt")

✅ File saved successfully at: c:\Users\linoc\OneDrive\Encoder\03_partos\02_scripts\Premature_model\data\interim\encoding_map_CSALUD01.txt
